# PARC2026 — 69c OpenVLA streaming bridge validation

69bでfull RLDSの推定容量が **56.61 GiB > 35 GiB** となったため、全量RLDSを複製せず、D10-selected `V2_SQRT_BALANCED_RAW` をLeRobotから直接OpenVLAへstreamingする経路を検証します。

このNotebookは学習を開始しません。選択済み10,758 episode / 1,620,614 framesのaction・stateだけを走査してOpenVLA用q01/q99統計を作り、69bと同じ先頭8 episodeをdirect streamingで再読込して画像・言語・7D action・8D proprio・8-action chunkを検証します。

- D10再選択・manifest再生成はしません。
- full RLDS / 56.6 GiBの複製は作りません。
- OpenVLA-OFT revisionは `e4287e94541f459edc4feabc4e181f537cd569a8` 固定です。
- LIBERO contract: action=7D, proprio=8D, chunk=8, normalization=`bounds_q99`。
- gripperは `1 - clip(raw_action[6], 0, 1)`、movement 6Dのみq01/q99正規化、gripperはabsolute・非正規化です。
- PASS時はDriveに `openvla-streaming-selected-v1/streaming_bridge_contract.json` を保存します。
- 69c PASS後もNotebook 70はcontroller gate確認だけで、M3 trainingはまだ自動開始しません。

**操作:** 下のコードセル1つだけを実行し、`=== 69c COMPLETE ===` を確認してください。


In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_69c_recovery'
PIN = '734f8c4077deb4ca4b09c30b34303445b5cc1edb'
URL = 'https://github.com/yu37330/py_AI.git'

if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('69c validation code:', got, flush=True)

DRIVE = Path('/content/drive/MyDrive/parc2026-cache')
SOURCE = DRIVE / 'datasets/lerobot_libero_plus_v3_train'
MANIFEST = DRIVE / 'pi05-ablation-group-aware-v2/dataset_ablation_manifests_v2_group_aware/V2_SQRT_BALANCED_RAW.json'
OUT69B = DRIVE / 'openvla-rlds-selected-v1'
REPORT69B = OUT69B / 'bridge_smoke_report.json'
CAP69B = OUT69B / 'bridge_capacity_decision.json'
OUT = DRIVE / 'openvla-streaming-selected-v1'
CONTRACT = OUT / 'streaming_bridge_contract.json'

for p in (SOURCE / 'meta/info.json', MANIFEST, REPORT69B, CAP69B):
    if not p.is_file():
        raise FileNotFoundError(p)

def env_ok(py):
    if not py.is_file():
        return False
    check = (
        "import sys,numpy,pandas,pyarrow,av;"
        "assert sys.version_info[:2]==(3,10);"
        "assert av.__version__=='12.3.0'"
    )
    return subprocess.run(
        [str(py), '-c', check],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode == 0

PY = ROOT / 'venv-openvla-rlds/bin/python'
if not env_ok(PY):
    VENV = ROOT / 'venv-openvla-streaming'
    PY = VENV / 'bin/python'
    uv = shutil.which('uv')
    if not uv:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
        uv = shutil.which('uv') or str(Path(sys.executable).parent / 'uv')
    subprocess.run([uv, 'python', 'install', '3.10'], check=True)
    if not PY.is_file():
        subprocess.run([uv, 'venv', '--python', '3.10', str(VENV)], check=True)
    subprocess.run([
        uv, 'pip', 'install', '--python', str(PY),
        '--only-binary', ':all:',
        'numpy<2', 'pandas>=2,<3', 'pyarrow>=16', 'av==12.3.0'
    ], check=True)
    if not env_ok(PY):
        raise RuntimeError('69c streaming environment verification failed')
else:
    print('[reuse] 69b Python 3.10 data environment', flush=True)

OUT.mkdir(parents=True, exist_ok=True)
status_path = OUT / 'streaming_bridge_status.json'
status_path.write_text(json.dumps({
    'schema_version': 1,
    'stage': '69c',
    'status': 'RUNNING',
    'code_pin': PIN,
}, indent=2) + '\n', encoding='utf-8')

cmd = [
    str(PY), '-u',
    str(REPO / 'tools/data/validate_lerobot_openvla_streaming_bridge.py'),
    '--lerobot-root', str(SOURCE),
    '--manifest', str(MANIFEST),
    '--69b-report', str(REPORT69B),
    '--69b-capacity', str(CAP69B),
    '--out', str(CONTRACT),
]
try:
    subprocess.run(cmd, check=True)
except subprocess.CalledProcessError as exc:
    status_path.write_text(json.dumps({
        'schema_version': 1,
        'stage': '69c',
        'status': 'FAILED',
        'code_pin': PIN,
        'error': str(exc),
    }, indent=2) + '\n', encoding='utf-8')
    print('\n=== 69c DIAGNOSTICS ===')
    print(status_path.read_text(encoding='utf-8'))
    raise

contract = json.loads(CONTRACT.read_text(encoding='utf-8'))
if contract.get('status') != 'PASS' or contract.get('bridge_type') != 'lerobot_streaming':
    raise RuntimeError(f'69c contract is not PASS: {contract}')
status_path.write_text(json.dumps({
    'schema_version': 1,
    'stage': '69c',
    'status': 'PASS',
    'code_pin': PIN,
    'contract': str(CONTRACT),
}, indent=2) + '\n', encoding='utf-8')
print('=== 69c COMPLETE ===', flush=True)
print('Contract:', CONTRACT, flush=True)
print('Next: verify Notebook 70 controller gate; do not start M3 training automatically.', flush=True)
